# Exploratory Data Analysis — Betting Platform Customer Churn

This notebook explores the customer dataset generated from real odds data (The Odds API) and simulated betting behavior. We analyze churn distribution, customer segments, and behavioral patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

df = pd.read_csv("../data/processed/customer_features.csv")
txns = pd.read_csv("../data/raw/transactions.csv", parse_dates=["bet_date"])
print(f"Customers: {len(df)} | Transactions: {len(txns)}")
df.head()

## 1. Dataset Overview

In [ ]:
print("Shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nNumeric summary:")
df.describe().round(2)

## 2. Churn Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count
churn_counts = df["churned"].value_counts()
axes[0].bar(["Retained", "Churned"], churn_counts.values, color=["#2ecc71", "#e74c3c"])
axes[0].set_title("Customer Churn Count")
axes[0].set_ylabel("Number of Customers")
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 20, str(v), ha="center", fontweight="bold")

# Rate
axes[1].pie(
    churn_counts.values,
    labels=["Retained", "Churned"],
    colors=["#2ecc71", "#e74c3c"],
    autopct="%1.1f%%",
    startangle=90,
    textprops={"fontsize": 13},
)
axes[1].set_title("Churn Rate")

plt.tight_layout()
plt.savefig("../data/processed/churn_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Churn by Customer Segments

In [ ]:
cat_cols = ["preferred_sport", "bet_type_preference", "deposit_method", "gender", "state"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)["churned"].mean().sort_values(ascending=False)
    churn_rate.plot(kind="bar", ax=axes[i], color="#e74c3c", alpha=0.8)
    axes[i].set_title(f"Churn Rate by {col.replace('_', ' ').title()}")
    axes[i].set_ylabel("Churn Rate")
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis="x", rotation=45)

axes[-1].axis("off")
plt.tight_layout()
plt.savefig("../data/processed/churn_by_segment.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Behavioral Patterns — Churned vs Retained

In [ ]:
num_cols = [
    "total_bets", "avg_stake", "total_pnl", "win_rate",
    "days_since_last_bet", "bet_frequency", "support_tickets", "tenure_days"
]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for label, color in [(0, "#2ecc71"), (1, "#e74c3c")]:
        subset = df[df["churned"] == label][col].dropna()
        axes[i].hist(subset, bins=30, alpha=0.6, color=color,
                     label="Churned" if label else "Retained")
    axes[i].set_title(col.replace("_", " ").title())
    axes[i].legend()

plt.tight_layout()
plt.savefig("../data/processed/behavior_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Correlation Heatmap

In [ ]:
corr_cols = [
    "tenure_days", "total_bets", "total_staked", "avg_stake", "total_pnl",
    "win_rate", "days_since_last_bet", "bet_frequency", "support_tickets",
    "promo_bets_pct", "unique_sports", "churned"
]

plt.figure(figsize=(14, 10))
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn_r",
            center=0, square=True, linewidths=0.5)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("../data/processed/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Inactivity vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Days since last bet distribution
sns.boxplot(data=df, x="churned", y="days_since_last_bet", ax=axes[0],
            palette={0: "#2ecc71", 1: "#e74c3c"})
axes[0].set_xticklabels(["Retained", "Churned"])
axes[0].set_title("Days Since Last Bet")

# P&L vs churn
sns.boxplot(data=df, x="churned", y="total_pnl", ax=axes[1],
            palette={0: "#2ecc71", 1: "#e74c3c"})
axes[1].set_xticklabels(["Retained", "Churned"])
axes[1].set_title("Total Profit/Loss")

plt.tight_layout()
plt.savefig("../data/processed/inactivity_pnl_churn.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Monthly Betting Activity Trend

In [ ]:
txns["month"] = txns["bet_date"].dt.to_period("M")
monthly = txns.groupby("month").agg(
    total_bets=("stake", "count"),
    total_staked=("stake", "sum"),
    unique_bettors=("customer_id", "nunique"),
).reset_index()
monthly["month"] = monthly["month"].astype(str)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.bar(monthly["month"], monthly["total_bets"], color="#3498db", alpha=0.7, label="Total Bets")
ax1.set_ylabel("Total Bets", color="#3498db")
ax1.tick_params(axis="x", rotation=45)

ax2 = ax1.twinx()
ax2.plot(monthly["month"], monthly["unique_bettors"], color="#e74c3c", marker="o", linewidth=2, label="Active Bettors")
ax2.set_ylabel("Active Bettors", color="#e74c3c")

plt.title("Monthly Betting Activity")
fig.tight_layout()
plt.savefig("../data/processed/monthly_activity.png", dpi=150, bbox_inches="tight")
plt.show()